<a href="https://colab.research.google.com/github/ericb42/GB885-Final-Brauer-E/blob/main/GB885_Final_Brauer_E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project for GB885 - Python Fundamentals

# RUSH Case Study
RUSH is a globally renowned sportswear and footwear brand.

This notebook is meant to provide analysis of RUSH sales data for use in understanding the market and identifying opportunities for growth.

## Import modules / packages and load data files

In [ ]:
# import modules and packages
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import seaborn as sns

In [ ]:
# load data files
products_url = 'https://raw.githubusercontent.com/ericb42/GB885-Final-Brauer-E/refs/heads/main/data/TABLE_PRODUCTS_885.csv'
retailer_url = 'https://raw.githubusercontent.com/ericb42/GB885-Final-Brauer-E/refs/heads/main/data/TABLE_RETAILER_885.csv'
sales_url = 'https://raw.githubusercontent.com/ericb42/GB885-Final-Brauer-E/refs/heads/main/data/TABLE_SALES_885.csv'

# products data (pipe separated)
products_df = pd.read_csv(products_url, sep = '|')

# retailer data (comma separated)
retailer_df = pd.read_csv(retailer_url)

# sales data (comma separated)
sales_df = pd.read_csv(sales_url)

## Preview the data sets

### Products

In [ ]:
# preview products_df
products_df.head()

In [ ]:
products_df.info()

### Retailers

In [ ]:
# preview retailer_df
retailer_df.head()

In [ ]:
retailer_df.info()

### Sales

In [ ]:
# preview sales_df
sales_df.head()

In [ ]:
sales_df.info()

## Initial clean up of data sets
When originally consolidating the dataframes, we found that the number of records increased from 9,648 to 10,271. This occurred because the RETAILER_ID column in retailer_df, which is identified as the primary key, contains duplicate values. When a sales record matches one of these duplicated RETAILER_ID values, the merge produces multiple rows for that sale.  

Because there is no reliable way to determine which retailer a duplicated RETAILER_ID should reference, we will not attempt to assign those sales to a specific retailer. However, since the business questions focus primarily on state-level analysis, we can preserve the validity of those analyses by removing only the retailer information for the affected records while retaining their state and city information.  

One duplicated RETAILER_ID presents the opposite situation: the retailer information is distinguishable, but the state and city information are identical and therefore cannot be uniquely assigned. For this case, we will remove one of the duplicate RETAILER_ID records and clear the STATE and CITY values from the remaining record. This preserves the retailer information while preventing incorrect geographic attribution.  

This approach maximizes the amount of usable data while avoiding assumptions about which retailer or geographic location should be associated with any ambiguous sales record.

In [ ]:
# identify retailer_df duplicates

retailer_df[retailer_df.duplicated(subset=['RETAILER_ID'], keep=False)].sort_values('RETAILER_ID')

In [ ]:
# create list of duplicate indices
duplicate_indicies = [63,81,84,83]

# drop records for duplicate_indicies
retailer_dedup_df = retailer_df.drop(duplicate_indicies)

# check our work
retailer_dedup_df.info()
retailer_dedup_df[retailer_dedup_df.duplicated(subset=['RETAILER_ID'], keep=False)].sort_values('RETAILER_ID')

In [ ]:
# remove the RETAILER info for the applicable ambiguous RETAILER_ID records
amb_retailer_id = ['W00SARLI', 'W00SFLOR', 'W00STEHO']
retailer_dedup_df.loc[retailer_dedup_df['RETAILER_ID'].isin(amb_retailer_id), ['RETAILER']] = ''

# check our work
retailer_dedup_df.loc[retailer_dedup_df['RETAILER_ID'].isin(amb_retailer_id)]

In [ ]:
# remove STATE and CITY info for the applicable ambiguous RETAILER_ID records
amb_retailer_id = ['S00NNENE']
retailer_dedup_df.loc[retailer_dedup_df['RETAILER_ID'].isin(amb_retailer_id), ['STATE', 'CITY']] = ''

# check our work
retailer_dedup_df.loc[retailer_dedup_df['RETAILER_ID'].isin(amb_retailer_id)]

## Consolidate the dataframes

In [ ]:
# merge sales data and retailer data based on RETAILER_ID
sales_retailer_df = pd.merge(sales_df, retailer_dedup_df, on = 'RETAILER_ID', how = 'left')

# check our work
sales_retailer_df.head()

In [ ]:
sales_retailer_df.info()

In [ ]:
# merge new sales_retailer_df with products_df using PRODUCT_ID
sales_retailer_products_df = pd.merge(sales_retailer_df, products_df, on = 'PRODUCT_ID', how = 'left')

# create srp_df reference to sales_retailer_products_df for more concise coding
srp_df = sales_retailer_products_df

# check our work
srp_df.head()

In [ ]:
srp_df.info()

## Inspect the Data

### Check for null values

In [ ]:
# Traditional null values
srp_df.isnull().sum()

Findings:
 - (2) null values in PRICE_PER_UNIT
 - (1) null value in RETAILER
 - (1) null value in REGION
 - (1) null value in STATE
 - (1) null value in CITY

In [ ]:
# non-traditional null values: categorical data
# list of categorical variables in dataframe
cat_var = list(srp_df.select_dtypes(include = ['object']).columns)

for column in cat_var:
  print(column)
  print(srp_df[column].unique())

Findings:
 - There is a '999999999' in RETAILER_ID
 - There is a '***' in UNITS_SOLD

In [ ]:
# non-traditional null values: numerical data
srp_df.describe()

Findings:
 - 99999.000000 in PRICE_PER_UNIT




### Check for duplicate values

In [ ]:
# check for duplicates
srp_df.duplicated().sum()

### Check for erroneous values

In [ ]:
# check value count for erroneous data
# list categorical variables
cat_var = list(srp_df.select_dtypes(include = ['object']).columns)

# set pandas option so that all rows are displayed
pd.set_option('display.max_rows', None)

for column in cat_var:
  print(column)
  print(srp_df[[column]].value_counts())

Findings:  
Again we see a record with RETAILER_ID of 999999999, we see that there is a '***' in UNITS_SOLD, and we also see the misspelling of 'outlet'.

### Check for outliers

Before we can check for outliers in units sold, we need to remove the '***' values and convert it to numeric.

In [ ]:
# Remove records with '***' in UNITS_SOLD
srp_df = srp_df[srp_df['UNITS_SOLD'] != '***']

# Convert UNITS_SOLD to numeric
srp_df['UNITS_SOLD'] = pd.to_numeric(srp_df['UNITS_SOLD'])

# check our work by making sure we dropped 2 records and that UNITS_SOLD is now numeric
srp_df.info()

In [ ]:
# use IQR method
# function to calculate IQR and print rows with values that fall outside that IQR
def count_iqr_outliers(df, column):
    # define q1
    q1 = df[column].quantile(0.25)
    # define q3
    q3 = df[column].quantile(0.75)
    # define iqr
    iqr = q3 - q1
    # define outlier thresholds
    l_threshold = q1 - 1.5 * iqr
    u_threshold = q3 + 1.5 * iqr
    # dount outliers
    outliers = (df[column] < l_threshold) | (df[column] > u_threshold)
    # Count the number of True values (outliers)
    return outliers.sum()

In [ ]:
# columns to check for outliers:
num_var = list(['PRICE_PER_UNIT', 'UNITS_SOLD', 'OPERATING_MARGIN'])

In [ ]:
for column in num_var:
  print(f'{column} : {count_iqr_outliers(srp_df, column)}')

## Cleaning Data

### Handle missing values
Previously, we set some values to empty strings because we didn't want to create duplicates. Now we'll put 'unknown' in those empty strings to make it more obvious that the data is unavailable rather than simply missing.

In [ ]:
# check RETAILER, CITY, and STATE columns for empty string and replace with 'unknown'
srp_df.loc[:, 'RETAILER'] = srp_df['RETAILER'].replace('', 'unknown')
srp_df.loc[:, 'CITY'] = srp_df['CITY'].replace('', 'unknown')
srp_df.loc[:, 'STATE'] = srp_df['STATE'].replace('', 'unknown')

# display modified rows to check our work
srp_df.loc[srp_df['RETAILER'] == '']
srp_df.loc[srp_df['CITY'] == '']
srp_df.loc[srp_df['STATE'] == '']

In [ ]:
srp_df.info()

The null values in RETAILER, REGION, STATE, and CITY are associated with the record that has RETAILER_ID = 999999999. We can eliminate these null values by removing this record.

In [ ]:
# remove the row with RETAILER_ID = 999999999
srp_df = srp_df[srp_df['RETAILER_ID'] != '999999999']

# check our work
srp_df.info()

The two records with null values in PRICE_PER_UNIT both have PRODUCT_ID of 20. We can take the mean PRICE_PER_UNIT of all sales with PRODUCT_ID of 20 and then use that value to replace the null values.

In [ ]:
# calculate the mean PRICE_PER_UNIT for all sales with PRODUCT_ID of 20
mean_price = srp_df.loc[srp_df['PRODUCT_ID'] == 20, 'PRICE_PER_UNIT'].mean()

# replace the null values
srp_df.loc[srp_df['PRODUCT_ID'] == 20, 'PRICE_PER_UNIT'] = srp_df.loc[srp_df['PRODUCT_ID'] == 20, 'PRICE_PER_UNIT'].fillna(mean_price)

# check out work
srp_df.info()

There is an additional record with a PRICE_PER_UNIT of 99999, a value which may have been entered because the actual price was not known, or it may be erroneous data. In either event, it is also PRODUCT_ID 20, so we will also replace this with the mean_price.

In [ ]:
# replace PRICE_PER_UNIT of 99999 with mean_price
srp_df.loc[srp_df['PRICE_PER_UNIT'] == 99999, 'PRICE_PER_UNIT'] = mean_price

# check our work
srp_df.describe()

### Handle erroneous values

In [ ]:
# replace Ootlet with Outlet
srp_df.replace(to_replace='Ootlet', value='Outlet', inplace=True)

# check our work
print(srp_df[['SALES_METHOD']].value_counts())

### Handle outliers
With the correction of the record of PRICE_PER_UNIT of 99999, there are no outliers beyond the 5th and 95th percentile, so we will make not reject any of the data.

## Answer the Business Questions

### Question 1:

### What product category had the highest sales (in dollars) in 2021? Whow much did it sell?

In [ ]:
# create a new dataframe so that we can perform analysis witout altering our cleaned data
rush_df = srp_df.copy()

rush_df.info()

In [ ]:
rush_df.head()

In [ ]:
# create and implement a filter that uses data from YEAR 2021 only
rush_2021_df = rush_df[rush_df['YEAR'] == 2021]

# check our work
rush_2021_df.head()

In [ ]:
# calculate product sales by using PRICE_PER_UNIT * UNITS_SOLD, grouped by PRODUCT_ID
sales_sum = (rush_2021_df['PRICE_PER_UNIT'] * rush_2021_df['UNITS_SOLD']).groupby([rush_2021_df['PRODUCT_ID'], rush_2021_df['PRODUCT_NAME']]).sum().reset_index().rename(columns={0: 'TOTAL_SALES'})

# set pandas options to not use scientific values
pd.set_option('display.float_format', '{:.2f}'.format)

# sort the sales_sum data
sales_sum = sales_sum.sort_values(by='TOTAL_SALES', ascending=False)

# display the product sales
print (sales_sum)

In [ ]:
# Create a plot showing sales by product for 2021
plt.figure(figsize=(8, 4))
sns.set_theme(style="darkgrid")
sns.barplot(
    data=sales_sum,
    x="TOTAL_SALES",
    y="PRODUCT_NAME",
)

# set x-axis to standard notation
plt.gca().xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

plt.title("Total Sales by Product Name (2021)")
plt.xlim(0, 25000000)

#### Answer 1:
#### Men's Street Footwear has the highest total sales, with $22,686,892.71

### Question 2:

### What state had the highest sales (in dollars) of women's products in 2021? How much was it?

In [ ]:
# add a filter for womens products to the data that is already filtered to year 2021
womens_2021_df = rush_2021_df[rush_2021_df['PRODUCT_NAME'].str.contains('Women', na=False)]

# check our work
womens_2021_df.head()

In [ ]:
# calculate product sales by using PRICE_PER_UNIT * UNITS_SOLD, grouped by STATE
sales_sum = (womens_2021_df['PRICE_PER_UNIT'] * womens_2021_df['UNITS_SOLD']).groupby([womens_2021_df['STATE']]).sum().reset_index().rename(columns={0: 'TOTAL_SALES'})

# sort the sales_sum
sales_sum = sales_sum.sort_values(by='TOTAL_SALES', ascending=False)

# display the product sales
print (sales_sum)

In [ ]:
# Create a plot showing sales by state for women's products for 2021

# only display the top 5 states
sales_sum_top_5 = sales_sum.nlargest(5, 'TOTAL_SALES')

print(sales_sum_top_5)
plt.figure(figsize=(6, 4))
sns.set_theme(style="darkgrid")
sns.barplot(
    data=sales_sum_top_5,
    x="TOTAL_SALES",
    y="STATE",
)

# set x-axis to use standard notation
plt.gca().xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
plt.title("Women's Top 5 Total Sales by State (2021)")
plt.xlim(0, 2500000)

#### Answer 2:
#### Maine has the highest sales in women's products for 2021 with $2,176,301.00

### Question 3:
### What state had the highest sales (in dollars) of men's products in 2021? How much was it?

In [ ]:
# add a filter for mens products to the data that is already filtered to year 2021
mens_2021_df = rush_2021_df[rush_2021_df['PRODUCT_NAME'].str.contains('Men', na=False)]

# check our work
mens_2021_df.head()

In [ ]:
# calculate product sales by using PRICE_PER_UNIT * UNITS_SOLD, grouped by STATE
sales_sum = (mens_2021_df['PRICE_PER_UNIT'] * mens_2021_df['UNITS_SOLD']).groupby([mens_2021_df['STATE']]).sum().reset_index().rename(columns={0: 'TOTAL_SALES'})

# sort the sales_sum
sales_sum = sales_sum.sort_values(by='TOTAL_SALES', ascending=False)

# display the product sales
print (sales_sum)

In [ ]:
# Create a plot showing sales by state for women's products for 2021

# only display the top 5 states
sales_sum_top_5 = sales_sum.nlargest(5, 'TOTAL_SALES')

print(sales_sum_top_5)
plt.figure(figsize=(6, 4))
sns.set_theme(style="darkgrid")
sns.barplot(
    data=sales_sum_top_5,
    x="TOTAL_SALES",
    y="STATE",
)

# set x-axis to standard notation
plt.gca().xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

plt.title("Men's Top 5 Total Sales by State (2021)")
plt.xlim(0, 2500000)

#### Answer 3:
#### Delaware has the highest sales in men's products for 2021 with $2,334,300.00

### Question 4:
### What retailer purchased the most units in 2021? In 2020?

In [ ]:
# create and implement a filter that uses data from YEAR 2020 only
rush_2020_df = rush_df[rush_df['YEAR'] == 2020]

# check our work
rush_2020_df.head()

In [ ]:
# determine the sum of units ordered for 2021, grouped by retailer
units_sum_2021 = rush_2021_df['UNITS_SOLD'].groupby([rush_2021_df['RETAILER']]).sum().reset_index()

# sort the units_sum_2021 data
units_sum_2021 = units_sum_2021.sort_values(by='UNITS_SOLD', ascending=False)

# display the results
print (units_sum_2021)

In [ ]:
# Create a plot showing sales by retailer for 2021
plt.figure(figsize=(6, 4))
sns.set_theme(style="darkgrid")
sns.barplot(
    data=units_sum_2021,
    x="UNITS_SOLD",
    y="RETAILER",
)

# set x-axis to use standard notation
plt.gca().xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

plt.title("Total Units Sold by Retailer (2021)")
plt.xlim(0, 1150000)

In [ ]:
# determine the sum of units ordered for 2020, grouped by retailer
units_sum_2020 = rush_2020_df['UNITS_SOLD'].groupby([rush_2020_df['RETAILER']]).sum().reset_index()

# sort the units_sum_2020 data
units_sum_2020 = units_sum_2020.sort_values(by='UNITS_SOLD', ascending=False)

# display the results
print (units_sum_2020)

In [ ]:
# Create a plot showing sales by retailer for 2021
plt.figure(figsize=(6, 4))
sns.set_theme(style="darkgrid")
sns.barplot(
    data=units_sum_2020,
    x="UNITS_SOLD",
    y="RETAILER",
)

# set x-axis to use standard notation
plt.gca().xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

plt.title("Total Units Sold by Retailer (2020)")
plt.xlim(0, 1150000)

#### Answer 4:
#### The retailer with the most units purchased in 2021 was Foot Locker, with 1,097,410 units.
#### For 2020, it was Amazon, with 317,930 units.

## Additional Analysis

### Sales per month


We'll start by looking at all sales on a monthly basis for the full range of the dataset.

In [ ]:
# Create a copy of the dataframe
rush2_df = rush_df.copy()

# Combine YEAR and MONTH into a standard datetime column
rush2_df["PLOT_DATE"] = pd.to_datetime(
    rush2_df["YEAR"].astype(str) + "-" + rush2_df["MONTH"].astype(str) + "-01"
)

# Group by the new PLOT_DATE and sum the UNITS_SOLD
monthly_sales = (
    rush2_df.groupby("PLOT_DATE")["UNITS_SOLD"].sum().reset_index()
)

# Create the plot
sns.set_theme(style="darkgrid")
plt.figure(figsize=(12, 6))

sns.lineplot(data=monthly_sales, x="PLOT_DATE", y="UNITS_SOLD", marker="o")

# Customize labels
plt.title("Total Units Sold Per Month Over Time")
plt.xlabel("Date")
plt.ylabel("Total Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Sales per month, by retailer:

We can also look at the same data, but grouped by RETAILER

In [ ]:
# Group by the PLOT_DATE and sum the UNITS_SOLD
monthly_sales = (
    rush2_df.groupby(["PLOT_DATE", "RETAILER"])["UNITS_SOLD"].sum().reset_index()
)

# Create the plot
sns.set_theme(style="darkgrid")
plt.figure(figsize=(12, 6))

sns.lineplot(data=monthly_sales, x="PLOT_DATE", y="UNITS_SOLD", hue="RETAILER", marker="o")

# Customize labels
plt.title("Total Units Sold Per Month Over Time, By Retailer")
plt.xlabel("Date")
plt.ylabel("Total Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Some retailers only have data for 2021. There also appears to be an up-tick in UNITS_SOLD for all retailers going into 2021.

### Sales per month by sales method:

Looking at sales over time by sales method, all methods saw a noticeable jump going into 2021, but growth in online sales outpaced other methods.

In [ ]:
# Group by the new PLOT_DATE and sum the UNITS_SOLD
monthly_sales = (
    rush2_df.groupby(["PLOT_DATE", "SALES_METHOD"])["UNITS_SOLD"].sum().reset_index()
)

# Create the plot
sns.set_theme(style="darkgrid")
plt.figure(figsize=(12, 6))

sns.lineplot(data=monthly_sales, x="PLOT_DATE", y="UNITS_SOLD", hue="SALES_METHOD", marker="o")

# Customize labels
plt.title("Total Units Sold Per Month Over Time, By Sales Method")
plt.xlabel("Date")
plt.ylabel("Total Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Sales per month by region (2021 only)

We can look at UNITS_SOLD per month by REGION.  

The South and Midwest follow roughly the same pattern, with peaks in July and troughs at around March and October.  

Midwest, West, and Northeast seem like they may also follow a somewhat similar pattern, with high points around May and September, and low points around June and November.

In [ ]:
# Group by PLOT_DATE and sum the UNITS_SOLD
monthly_sales = (
    rush2_df.groupby(["PLOT_DATE", "REGION"])["UNITS_SOLD"].sum().reset_index()
)

# Create the plot
sns.set_theme(style="darkgrid")
plt.figure(figsize=(12, 10))

# only show data for 2021
sns.lineplot(data=monthly_sales[(monthly_sales["PLOT_DATE"] >= "2021-01") & (monthly_sales["PLOT_DATE"] < "2022-01")], x="PLOT_DATE", y="UNITS_SOLD", hue="REGION", marker="o")

# Customize labels
plt.title("Total Units Sold Per Month Over Time, By Region")
plt.xlabel("Date")
plt.ylabel("Total Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Sales per month by product name (2021 only)

When looking at data for 2021, Units Sold per month by Product Name, all product appear to follow a similar pattern.

In [ ]:
# Group by the PLOT_DATE and sum the UNITS_SOLD
monthly_sales = (
    rush2_df.groupby(["PLOT_DATE", "PRODUCT_NAME"])["UNITS_SOLD"].sum().reset_index()
)

# Create the plot
sns.set_theme(style="darkgrid")
plt.figure(figsize=(10, 8))

# only show data for 2021
sns.lineplot(data=monthly_sales[(monthly_sales["PLOT_DATE"] >= "2021-01") & (monthly_sales["PLOT_DATE"] < "2022-01")], x="PLOT_DATE", y="UNITS_SOLD", hue="PRODUCT_NAME", marker="o")

# Customize labels
plt.title("Total Units Sold Per Month Over Time, By Product Name")
plt.xlabel("Date")
plt.ylabel("Total Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()